# REQ ↔ TEST Mapping — Multi-Agent Pipeline

| Cell | Block | What it does |
|------|-------|--------------|
| 1 | Imports | All libraries |
| 2 | API & Agent Init | Credentials from `.env`, LLM + LangChain chain setup |
| 3 | Variables | File paths, batch size |
| 4 | Agent 1 — Prompt Agent | AI extracts tokens, activation words, noise → new prompt |
| 5 | Agent 2 — REQ-TEST Agent | Maps reqs → test cases using Agent 1 output |
| 6 | Excel Report | Light-themed styled output |

In [21]:
# ═══════════════════════════════════════════════════════════════════
#  CELL 0 — Install dependencies (skips packages already installed)
# ═══════════════════════════════════════════════════════════════════
import subprocess, sys
from importlib.util import find_spec

# package to import-name mapping (they sometimes differ)
packages = {
    'pandas':          'pandas',
    'openpyxl':        'openpyxl',
    'python-docx':     'docx',
    'python-dotenv':   'dotenv',
    'httpx':           'httpx',
    'langchain':       'langchain',
    'langchain-openai':'langchain_openai',
    'langchain-core':  'langchain_core',
    'ipywidgets':      'ipywidgets',
}

for pkg, import_name in packages.items():
    if find_spec(import_name) is not None:
        print(f'  ✓ already installed  {pkg}')
    else:
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--quiet', pkg],
            capture_output=True, text=True
        )
        status = '✓ installed' if result.returncode == 0 else '✗ FAILED'
        print(f'  {status}          {pkg}')

print('\nDone.')

  ✓ already installed  pandas
  ✓ already installed  openpyxl
  ✓ already installed  python-docx
  ✓ already installed  python-dotenv
  ✓ already installed  httpx
  ✓ already installed  langchain
  ✓ already installed  langchain-openai
  ✓ already installed  langchain-core
  ✓ already installed  ipywidgets

Done.


---
## Cell 1 — Imports

In [14]:
# ── Standard library ──────────────────────────────────────────────────────────
import re
import time
import textwrap
from pathlib import Path

# ── Environment / secrets ─────────────────────────────────────────────────────
# python-dotenv loads your .env file so credentials never live in the notebook
from dotenv import load_dotenv
import os

# ── HTTP / proxy ──────────────────────────────────────────────────────────────
import httpx
from openai import DefaultHttpxClient

# ── Data & Excel ──────────────────────────────────────────────────────────────
import pandas as pd
from docx import Document
from openpyxl import Workbook
from openpyxl.styles import (
    Font, PatternFill, Alignment, Border, Side, GradientFill
)
from openpyxl.utils import get_column_letter

# ── Jupyter UI ────────────────────────────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── LangChain ─────────────────────────────────────────────────────────────────
# AzureChatOpenAI  : LangChain wrapper — gives .invoke() + chain compatibility
# ChatPromptTemplate: structures system/human message pairs as a reusable template
# StrOutputParser  : pulls plain string out of AIMessage response object
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print('✓ All imports OK')

✓ All imports OK


---
## Cell 2 — API & Agent Initialisation

Credentials are loaded from a `.env` file — nothing sensitive in the notebook.

```
# .env  (sits next to the notebook, never committed to git)
AZURE_API_KEY=your_key_here
AZURE_ENDPOINT=your_end_point
AZURE_API_VERSION=your_api_version
AZURE_DEPLOYMENT=your_model
PROXY_URL= proxy_url
```

In [ ]:
# ── Load .env ─────────────────────────────────────────────────────────────────
load_dotenv()  # reads .env from current directory

AZURE_API_KEY     = os.getenv('AZURE_API_KEY',     '')
AZURE_ENDPOINT    = os.getenv('AZURE_ENDPOINT',    '')
AZURE_API_VERSION = os.getenv('AZURE_API_VERSION', '')
AZURE_DEPLOYMENT  = os.getenv('AZURE_DEPLOYMENT',  '')
PROXY_URL         = os.getenv('PROXY_URL',         '')


# ── LangChain LLM ─────────────────────────────────────────────────────────────
# temperature=0 → deterministic, matches the prompt's DETERMINISTIC constraint
# max_tokens split:
#   Agent 1 (prompt builder) → 3000  (long structured prompt output)
#   Agent 2 (mapping)        → 4000  (full table for each batch)

llm_agent1 = AzureChatOpenAI(
    azure_endpoint=AZURE_ENDPOINT,
    azure_deployment=AZURE_DEPLOYMENT,
    api_version=AZURE_API_VERSION,
    api_key=AZURE_API_KEY,
    temperature=0,
    max_tokens=3000,
    http_client=DefaultHttpxClient(proxy=PROXY_URL),
)

llm_agent2 = AzureChatOpenAI(
    azure_endpoint=AZURE_ENDPOINT,
    azure_deployment=AZURE_DEPLOYMENT,
    api_version=AZURE_API_VERSION,
    api_key=AZURE_API_KEY,
    temperature=0,
    max_tokens=4000,
    http_client=DefaultHttpxClient(proxy=PROXY_URL),
)

print(f'✓ LLM agents initialised')
print(f'  Endpoint  : {AZURE_ENDPOINT}')
print(f'  Deployment: {AZURE_DEPLOYMENT}')
print(f'  API ver   : {AZURE_API_VERSION}')
print(f'  Proxy     : {PROXY_URL}')

✓ LLM agents initialised
  Endpoint  : https://openai-swc-pd-aura-001.openai.azure.com
  Deployment: gpt-4.1
  API ver   : 2024-12-01-preview
  Proxy     : http://localhost:9000


---
## Cell 3 — Variables: Input Files, Prompt File, Output

In [16]:
# ── Folder ────────────────────────────────────────────────────────────────────
INPUT_FOLDER = Path('InputFiles')

# ── Input files ───────────────────────────────────────────────────────────────
REQ_EXCEL_PATH  = INPUT_FOLDER / 'Requirements.xlsx'   # Requirements export
VV_EXCEL_PATH   = INPUT_FOLDER / 'VVtestcases.xlsx'   # VV Test Cases export 
PROMPT_DOC_PATH = INPUT_FOLDER / 'prompt_file.docx'    # Base prompt template

# ── Output file ───────────────────────────────────────────────────────────────
OUTPUT_PATH = INPUT_FOLDER / 'req_test_report.xlsx'    # Final mapping report

# ── Pipeline config ───────────────────────────────────────────────────────────
BATCH_SIZE = 10   # requirements per LLM call in Agent 2
                  # lower = more API calls but safer on token limits
                  # raise to 20 if your deployment allows 8k+ output tokens

# ── Validate all input files exist before running ─────────────────────────────
for p in [REQ_EXCEL_PATH, VV_EXCEL_PATH, PROMPT_DOC_PATH]:
    status = '✓' if p.exists() else '✗ MISSING'
    print(f'  {status}  {p}')

print(f'\n  Output → {OUTPUT_PATH}')
print(f'  Batch size: {BATCH_SIZE} reqs/call')

  ✓  InputFiles\Requirements.xlsx
  ✓  InputFiles\VVtestcases.xlsx
  ✓  InputFiles\prompt_file.docx

  Output → InputFiles\req_test_report.xlsx
  Batch size: 10 reqs/call


---
## Cell 4 — Agent 1: AI-Powered Prompt Builder

**What this agent does:**
1. Reads both Excel files and extracts raw text (summaries, descriptions, test names, pre-conditions)
2. Sends all that raw text to the LLM and asks it to:
   - Identify **activation keywords** (action words, controls, states) from the actual data
   - Identify **noise tokens** (generic folder labels, version tags, boilerplate)
   - Identify **domain words** (recurring technical terms that are domain-specific but not activation words)
3. Substitutes column names, type values, and all three token lists into the base prompt
4. Outputs the final ready-to-use system prompt for Agent 2

**Why AI for tokenisation?**  
The files are in German with automotive domain vocabulary. A regex/frequency approach (the old version) catches surface tokens but can't distinguish *noise* (`regressionstest`, `smoketest`) from *domain* (`heckklappe`, `wischen`) from *activation* (`betätigen`, `drücken`). The LLM understands the difference.

In [17]:
# ═════════════════════════════════════════════════════════════════════════════
#  HELPERS — file reading (used by both agents)
# ═════════════════════════════════════════════════════════════════════════════

def read_docx(path: str) -> str:
    """Read all non-empty paragraphs from a .docx file."""
    doc = Document(path)
    return '\n'.join(p.text for p in doc.paragraphs if p.text.strip())

def load_excel_export_sheet(path: str) -> pd.DataFrame:
    """
    Load the Export sheet.
    Row 0 = 'Report Generated on...' metadata  → skip with header=1
    Row 1 = real column headers                → become DataFrame columns
    """
    path  = str(path).strip()                          # ← guard whitespace/Path objects
    xls   = pd.ExcelFile(path, engine='openpyxl')      # ← force engine
    sheet = 'Export' if 'Export' in xls.sheet_names else xls.sheet_names[0]
    df    = pd.read_excel(xls, sheet_name=sheet, header=1, dtype=str).fillna('')
    df.columns = df.columns.str.strip()
    return df


def detect_columns(df: pd.DataFrame, patterns: dict) -> dict:
    """Find column names by matching lowercase patterns."""
    result = {}
    for key, candidates in patterns.items():
        found = next((c for c in df.columns
                      for cand in candidates if cand in c.lower()), candidates[0])
        result[key] = found
    return result


# ═════════════════════════════════════════════════════════════════════════════
#  AGENT 1 — AI-POWERED PROMPT BUILDER
# ═════════════════════════════════════════════════════════════════════════════

def prompt_agent(req_path, vv_path, docx_path):
    """
    Agent 1 — AI-Powered Prompt Builder.

    Step A: Extract raw corpus from both Excel files
    Step B: Ask LLM to classify tokens into 3 categories:
              activationKeywords, genericNoise, commonDomainWords
    Step C: Substitute columns + AI-generated token lists into base prompt
    Returns: final system prompt string for Agent 2
    """

    # ── Step A: Load files and detect columns ─────────────────────────────
    base_prompt = read_docx(docx_path)
    req_df      = load_excel_export_sheet(req_path)
    vv_df       = load_excel_export_sheet(vv_path)

    req_cols = detect_columns(req_df, {
        'id':   ['item id'],
        'sum':  ['summary'],
        'desc': ['description'],
        'type': ['type'],
    })
    vv_cols = detect_columns(vv_df, {
        'id':   ['item id'],
        'name': ['name'],
        'type': ['type'],
        'pre':  ['pre-condition', 'precondition'],
        'desc': ['description'],
    })

    req_types   = req_df[req_cols['type']].str.strip().unique().tolist()
    vv_types    = vv_df[vv_cols['type']].str.strip().unique().tolist()
    req_fol     = next((t for t in req_types if t.lower() == 'folder'), 'Folder')
    vv_fol      = next((t for t in vv_types  if t.lower() == 'folder'), 'Folder')
    vv_tc_type  = next((t for t in vv_types  if t.lower() in
                        ('verification', 'test case', 'testcase')), 'Verification')

    req_folders = req_df[req_df[req_cols['type']] == req_fol][req_cols['sum']].tolist()
    vv_folders  = vv_df[vv_df[vv_cols['type']]   == vv_fol ][vv_cols['name']].tolist()

    # ── Step B: Build the corpus for AI tokenisation ──────────────────────
    req_corpus = '\n'.join(
        req_df[req_cols['sum']].tolist() + req_df[req_cols['desc']].tolist()
    )[:6000]   # cap to stay within token budget

    vv_corpus = '\n'.join(
        vv_df[vv_cols['name']].tolist() + vv_df[vv_cols['pre']].tolist()
    )[:6000]

    folder_corpus = (
        'REQ FOLDERS:\n' + '\n'.join(req_folders) +
        '\n\nVV FOLDERS:\n' + '\n'.join(vv_folders)
    )

    # ── Step B: AI tokenisation call ──────────────────────────────────────
    # WHY LLM HERE instead of regex/frequency:
    #   German automotive vocabulary — the LLM understands context.
    #   It can tell that 'betätigen' is an activation word (action),
    #   'heckklappe' is a domain word (component), and
    #   'regressionstest' is noise (test management boilerplate).
    #   A frequency counter cannot make that distinction.

    tokenisation_prompt = ChatPromptTemplate.from_messages([
        ('system', """You are a domain-analysis agent for automotive requirements engineering.
You receive raw German text from two Excel exports (Requirements + VV Test Cases).
Your job is to classify the vocabulary into exactly 3 groups.

GROUP 1 — activationKeywords:
  Words that describe ACTIONS, CONTROLS, STATES, or TRIGGER CONDITIONS.
  Examples: betätigen, öffnen, schließen, drücken, tippwischen, klemme 15, aktiv, inaktiv
  Include: verbs of interaction, control element names, switch states, speed/voltage references

GROUP 2 — genericNoise (split into reqNoise and vvNoise):
  reqNoise: generic folder/document labels in the Requirements file that add no domain signal
  vvNoise:  generic folder/test labels in the VV file that add no domain signal
  Examples: funktionslastenheft, regressionstest, smoketest, sdv0.1, gv0.3, master, allgemein

GROUP 3 — commonDomainWords:
  Technical component/system nouns that appear frequently but are too broad to be activation keywords.
  Examples: heckklappe, frontscheibe, fahrzeug, system, funktion, tür, scheibe, antrieb

RULES:
- Output ONLY valid Python — a dict with 4 keys: activationKeywords, reqNoise, vvNoise, domainWords
- Each value is a Python list of lowercase strings
- No comments, no markdown fences, no explanation — just the dict literal
- Deduplicate. Min 10 items per list. Max 80 items per list.
- Use the actual words found in the corpus, do not invent terms"""),
        ('human', """REQ CORPUS (summaries + descriptions):
{req_corpus}

VV CORPUS (test names + pre-conditions):
{vv_corpus}

FOLDER NAMES:
{folder_corpus}

Return the Python dict now.""")
    ])

    tok_chain  = tokenisation_prompt | llm_agent1 | StrOutputParser()
    tok_raw    = tok_chain.invoke({
        'req_corpus':    req_corpus,
        'vv_corpus':     vv_corpus,
        'folder_corpus': folder_corpus,
    })

    # Parse the AI-returned dict safely
    try:
        clean = tok_raw.strip().removeprefix('```python').removeprefix('```').removesuffix('```').strip()
        tok   = eval(clean, {'__builtins__': {}})
        activation_keywords = tok.get('activationKeywords', [])
        req_noise           = tok.get('reqNoise', [])
        vv_noise            = tok.get('vvNoise', [])
        domain_words        = tok.get('domainWords', [])
    except Exception as e:
        print(f'  ⚠ Token parse failed ({e}), using empty lists')
        activation_keywords = req_noise = vv_noise = domain_words = []

    # ── Step C: Substitute everything into the base prompt ────────────────
    def fmt(lst): return '[' + ','.join(lst) + ']'

    preamble = (
        f'FILE DETECTION — auto by Prompt Agent (AI tokenisation):\n'
        f'  REQ  : {req_cols["id"]} | {req_cols["sum"]} | {req_cols["desc"]} | {req_cols["type"]}\n'
        f'  VV   : {vv_cols["id"]} | {vv_cols["name"]} | {vv_cols["type"]} | {vv_cols["pre"]} | {vv_cols["desc"]}\n'
        f'  Types: REQ={req_types}  VV={vv_types}\n'
        f'  activationKeywords ({len(activation_keywords)}): {activation_keywords[:8]}...\n'
        f'  reqNoise ({len(req_noise)}): {req_noise[:5]}...\n'
        f'  vvNoise  ({len(vv_noise)}): {vv_noise[:5]}...\n'
        f'  domainWords ({len(domain_words)}): {domain_words[:8]}...\n\n'
    )

    p = base_prompt

    # Column references
    p = p.replace(
        'Requirements (Export sheet): ID, Summary, Description, Type. Use ID not Foreign ID_int_ct.',
        f'Requirements (Export sheet): {req_cols["id"]}, {req_cols["sum"]}, {req_cols["desc"]}, {req_cols["type"]}. Use {req_cols["id"]} not Foreign ID_int_ct.'
    )
    p = p.replace(
        'VV Test Cases (Export sheet): ID, Name, Type, Pre-Condition, Description',
        f'VV Test Cases (Export sheet): {vv_cols["id"]}, {vv_cols["name"]}, {vv_cols["type"]}, {vv_cols["pre"]}, {vv_cols["desc"]}'
    )

    # Type values
    p = p.replace('Type=Test Case\u2192VVTestCases', f'Type={vv_tc_type}\u2192VVTestCases')
    p = p.replace('Type==Test Case',                f'Type=={vv_tc_type}')
    p = p.replace(
        'Type in [Functional,Non-Functional,Information]',
        f'Type in {fmt([t for t in req_types if t != req_fol])}'
    )

    # activationKeywords block
    ks = p.find('activationKeywords = [')
    ke = p.find('\ngenericReqFolderNoise=')
    if ks != -1 and ke != -1:
        p = p[:ks] + f'activationKeywords = {fmt(activation_keywords)}\n' + p[ke:]

    # genericReqFolderNoise
    rns = p.find('genericReqFolderNoise=[')
    rne = p.find(']\ngenericVVFolderNoise=[')
    if rns != -1 and rne != -1:
        p = p[:rns] + f'genericReqFolderNoise={fmt(req_noise)}\n' + p[rne+2:]

    # genericVVFolderNoise
    vns = p.find('genericVVFolderNoise=[')
    vne = p.find(']\ncommonDomainWords=[')
    if vns != -1 and vne != -1:
        p = p[:vns] + f'genericVVFolderNoise={fmt(vv_noise)}\n' + p[vne+2:]

    # commonDomainWords
    ds = p.find('commonDomainWords=[')
    de = p.find(']\nReqTokens=')
    if ds != -1 and de != -1:
        p = p[:ds] + f'commonDomainWords={fmt(domain_words)}\n' + p[de+2:]

    # Column name refs inside scoring logic
    p = p.replace('Summary+Description',             f'{req_cols["sum"]}+{req_cols["desc"]}')
    p = p.replace('Name+Pre-Condition+Description',  f'{vv_cols["name"]}+{vv_cols["pre"]}+{vv_cols["desc"]}')
    p = p.replace('ReqFolder.Summary',               f'ReqFolder.{req_cols["sum"]}')
    p = p.replace(
        '| Req ID | Req Description | Type(requiremnt) | Testcase ID | Testcase Name|',
        f'| {req_cols["id"]} | Req Description | Type | {vv_cols["id"]} | Testcase Name |'
    )

    return preamble + p, tok_raw


# ── Run Agent 1 ───────────────────────────────────────────────────────────────
print('Running Agent 1 — AI Prompt Builder...')
print('='*65)

FINAL_PROMPT, raw_tokens = prompt_agent(REQ_EXCEL_PATH, VV_EXCEL_PATH, PROMPT_DOC_PATH)

print('✓ Agent 1 complete\n')
print('── AI TOKEN CLASSIFICATION (raw LLM output) ─────────────────')
print(raw_tokens[:800], '...' if len(raw_tokens) > 800 else '')
print('\n── FINAL PROMPT (first 1500 chars) ──────────────────────────')
print(FINAL_PROMPT[:1500])
print(f'\n  Total prompt length: {len(FINAL_PROMPT)} chars')

Running Agent 1 — AI Prompt Builder...


c:\Users\UMH2CFQ\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\UMH2CFQ\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


✓ Agent 1 complete

── AI TOKEN CLASSIFICATION (raw LLM output) ─────────────────
{
  "activationKeywords": [
    "öffnen",
    "schließen",
    "betätigung",
    "tippwischen",
    "tipp-wischen",
    "aktivieren",
    "deaktivieren",
    "bedienung",
    "bedienstelle",
    "anzeige",
    "zustand",
    "stufe 1",
    "stufe 2",
    "dauerwischen",
    "intervallbetrieb",
    "intervallwischen",
    "regensensorbetrieb",
    "klemme 15",
    "ein",
    "aus",
    "verriegeln",
    "entriegeln",
    "verrieglung",
    "quittierung",
    "funkfernbedienung",
    "hmi",
    "mmi",
    "sprachbefehl",
    "sprachbedienung",
    "ausgangsposition",
    "parkposition",
    "parklage",
    "serviceposition",
    "servicestellung",
    "manuell",
    "automatisch",
    "überkrafterkennung",
    "überkraftbegrenzung",
    "timer",
    "anzeige",
    "hinweis",
    "information ...

── FINAL PROMPT (first 1500 chars) ──────────────────────────
FILE DETECTION — auto by Prompt Agent (AI tokenisa

---
## Cell 5 — Agent 2: REQ-TEST Mapping

Takes the prompt built by Agent 1 as its **system message**.
Processes requirements in batches of `BATCH_SIZE`, returns a DataFrame.

**LangChain role here:**  
`ChatPromptTemplate | llm_agent2 | StrOutputParser` — system prompt baked in once via `.partial()`, only the batch CSV changes per call.

In [18]:
# ═════════════════════════════════════════════════════════════════════════════
#  AGENT 2 — REQ-TEST MAPPING
# ═════════════════════════════════════════════════════════════════════════════

def parse_table(raw: str) -> list:
    """Parse LLM markdown pipe-table → list of 5-tuples."""
    rows = []
    for line in raw.splitlines():
        line = line.strip()
        if not line.startswith('|') or re.match(r'^\|[-| ]+\|$', line):
            continue
        cells = [c.strip() for c in line.strip('|').split('|')]
        if len(cells) >= 5 and cells[0].lower() not in ('req id','req_id','item id'):
            rows.append(tuple(cells[:5]))
    return rows


def req_test_agent(system_prompt: str, req_path, vv_path, batch_size=10):
    """
    Agent 2 — REQ-TEST Mapping Agent.

    LangChain RunnableSequence:
        ChatPromptTemplate.partial(system_prompt=...)  ← baked in once
        | llm_agent2                                   ← called per batch
        | StrOutputParser()                            ← extracts plain string
    """
    req_df = load_excel_export_sheet(req_path)
    vv_df  = load_excel_export_sheet(vv_path)

    req_typ = next((c for c in req_df.columns if c.lower() == 'type'), 'Type')
    req_df  = req_df[req_df[req_typ].str.strip() != 'Folder'].reset_index(drop=True)

    # Only send columns Agent 2 actually needs — reduces tokens per call
    vv_keep = [c for c in ['Item ID','Name','Type','Pre-Condition','Description','Parent']
               if c in vv_df.columns]
    vv_text = vv_df[vv_keep].to_csv(index=False)

    # Build chain — system prompt baked in via .partial()
    mapping_template = ChatPromptTemplate.from_messages([
        ('system', '{system_prompt}'),
        ('human',  '{user_message}'),
    ])
    chain = mapping_template.partial(system_prompt=system_prompt) | llm_agent2 | StrOutputParser()

    batches  = [req_df.iloc[i:i+batch_size] for i in range(0, len(req_df), batch_size)]
    all_rows = []
    total    = len(req_df)

    print(f'  Mapping {total} requirements in {len(batches)} batches...')

    for idx, batch_df in enumerate(batches):
        user_message = f"""Map all requirements in this batch to the best matching test cases.

REQUIREMENTS (batch {idx+1}/{len(batches)}):
{batch_df.to_csv(index=False)}

VV TEST CASES (full):
{vv_text}

Output ONLY a markdown table:
| Req ID | Req Description | Type | Testcase ID | Testcase Name |

One row per requirement. No match → Testcase ID = NO MATCH, Testcase Name = -
No commentary. No preamble. Table only."""

        raw    = chain.invoke({'user_message': user_message})
        parsed = parse_table(raw)
        all_rows.extend(parsed)
        print(f'  Batch {idx+1}/{len(batches)} done — {len(parsed)} rows mapped')

    cols = ['Req ID', 'Req Description', 'Type', 'Testcase ID', 'Testcase Name']
    return pd.DataFrame(all_rows, columns=cols) if all_rows else pd.DataFrame(columns=cols)


# ── Run Agent 2 ───────────────────────────────────────────────────────────────
print('Running Agent 2 — REQ-TEST Mapping...')
print('='*65)

RESULT_DF = req_test_agent(FINAL_PROMPT, REQ_EXCEL_PATH, VV_EXCEL_PATH, BATCH_SIZE)

print(f'\n✓ Agent 2 complete — {len(RESULT_DF)} rows')
print(f'  Matched  : {(RESULT_DF["Testcase ID"].str.upper() != "NO MATCH").sum()}')
print(f'  No match : {(RESULT_DF["Testcase ID"].str.upper() == "NO MATCH").sum()}')
print()
print(RESULT_DF.head(15).to_string(index=False))

Running Agent 2 — REQ-TEST Mapping...
  Mapping 93 requirements in 10 batches...


c:\Users\UMH2CFQ\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\UMH2CFQ\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


  Batch 1/10 done — 10 rows mapped
  Batch 2/10 done — 10 rows mapped
  Batch 3/10 done — 10 rows mapped
  Batch 4/10 done — 10 rows mapped
  Batch 5/10 done — 10 rows mapped
  Batch 6/10 done — 0 rows mapped
  Batch 7/10 done — 10 rows mapped
  Batch 8/10 done — 10 rows mapped
  Batch 9/10 done — 10 rows mapped
  Batch 10/10 done — 3 rows mapped

✓ Agent 2 complete — 83 rows
  Matched  : 29
  No match : 54

  Req ID                                                                                                                                                                                                    Req Description       Type Testcase ID Testcase Name
47420777                                                                                                                                                                                 Dummy Requirement for Excel Import Functional    NO MATCH             -
54359562                                                                  

---
## Cell 6 — Excel Report (light theme)

In [19]:
# ═════════════════════════════════════════════════════════════════════════════
#  EXCEL REPORT BUILDER — light professional theme
#
#  Colour palette (all ARGB hex, no alpha prefix needed for openpyxl fills):
#    Header bg   : 1F3864  (dark navy)       text: FFFFFF
#    Matched row : EBF3E8  (light green)     Testcase ID text: 2E7D32
#    No-match row: FDE8E8  (light red)       Testcase ID text: C62828
#    Alternating : F8F9FA  (very light grey)
#    Summary bg  : FFFFFF  borders: DEE2E6
# ═════════════════════════════════════════════════════════════════════════════

def build_report(df: pd.DataFrame, out_path: str) -> dict:

    wb = Workbook()

    # ─────────────────────────────────────────────────────────────────────
    #  SHEET 1 — REQ-TEST Mapping
    # ─────────────────────────────────────────────────────────────────────
    ws = wb.active
    ws.title = 'REQ-TEST Mapping'
    ws.sheet_view.showGridLines = False

    # Styles
    HDR_FILL     = PatternFill('solid', fgColor='1F3864')
    HDR_FONT     = Font(name='Calibri', bold=True, color='FFFFFF', size=11)
    HDR_ALIGN    = Alignment(horizontal='center', vertical='center', wrap_text=True)

    MATCH_FILL   = PatternFill('solid', fgColor='EBF3E8')
    NOMATCH_FILL = PatternFill('solid', fgColor='FDE8E8')
    ALT_FILL     = PatternFill('solid', fgColor='F8F9FA')

    MATCH_ID_FONT   = Font(name='Calibri', size=10, color='2E7D32', bold=True)
    NOMATCH_ID_FONT = Font(name='Calibri', size=10, color='C62828', bold=True)
    BODY_FONT       = Font(name='Calibri', size=10, color='212529')
    TYPE_FONT       = Font(name='Calibri', size=10, color='495057', italic=True)

    thin  = Side(style='thin',   color='DEE2E6')
    thick = Side(style='medium', color='1F3864')
    BORDER      = Border(left=thin, right=thin, top=thin, bottom=thin)
    HDR_BORDER  = Border(left=thick, right=thick, top=thick, bottom=thick)
    LEFT_ALIGN  = Alignment(horizontal='left', vertical='center', wrap_text=True)
    CTR_ALIGN   = Alignment(horizontal='center', vertical='center')

    headers    = ['Req ID', 'Req Description', 'Type', 'Testcase ID', 'Testcase Name']
    col_widths = [15, 55, 18, 16, 48]

    # Header row
    for ci, (h, w) in enumerate(zip(headers, col_widths), 1):
        cell           = ws.cell(row=1, column=ci, value=h)
        cell.font      = HDR_FONT
        cell.fill      = HDR_FILL
        cell.border    = HDR_BORDER
        cell.alignment = HDR_ALIGN
        ws.column_dimensions[get_column_letter(ci)].width = w
    ws.row_dimensions[1].height = 24

    # Data rows
    for ri, row in df.iterrows():
        excel_row = ri + 2
        is_match  = str(row.get('Testcase ID', '')).strip().upper() != 'NO MATCH'
        row_fill  = MATCH_FILL if is_match else NOMATCH_FILL
        if is_match and ri % 2 == 0:
            row_fill = ALT_FILL  # subtle alternating on matched rows

        for ci, col in enumerate(headers, 1):
            val            = row.get(col, '')
            cell           = ws.cell(row=excel_row, column=ci, value=val)
            cell.fill      = row_fill
            cell.border    = BORDER

            if col == 'Testcase ID':
                cell.font      = MATCH_ID_FONT if is_match else NOMATCH_ID_FONT
                cell.alignment = CTR_ALIGN
            elif col == 'Req ID':
                cell.font      = Font(name='Calibri', size=10, color='1F3864', bold=True)
                cell.alignment = CTR_ALIGN
            elif col == 'Type':
                cell.font      = TYPE_FONT
                cell.alignment = CTR_ALIGN
            else:
                cell.font      = BODY_FONT
                cell.alignment = LEFT_ALIGN

        ws.row_dimensions[excel_row].height = 20

    ws.freeze_panes = 'A2'

    # Auto-filter
    ws.auto_filter.ref = f'A1:{get_column_letter(len(headers))}1'

    # ─────────────────────────────────────────────────────────────────────
    #  SHEET 2 — Summary
    # ─────────────────────────────────────────────────────────────────────
    ws2 = wb.create_sheet('Summary')
    ws2.sheet_view.showGridLines = False

    total     = len(df)
    matched   = int((df['Testcase ID'].str.upper() != 'NO MATCH').sum())
    unmatched = total - matched
    pct       = round(matched / total * 100, 1) if total else 0

    # Title
    ws2.merge_cells('B2:E2')
    ws2['B2']           = 'REQ ↔ TEST Mapping Report'
    ws2['B2'].font      = Font(name='Calibri', bold=True, size=16, color='1F3864')
    ws2['B2'].alignment = Alignment(horizontal='left', vertical='center')
    ws2.row_dimensions[2].height = 30

    # Stat cards
    stats = [
        ('Total Requirements',  total,     '1F3864', 'E8EDF5'),
        ('Matched → Test Case', matched,   '2E7D32', 'EBF3E8'),
        ('No Match Found',      unmatched, 'C62828', 'FDE8E8'),
        ('Coverage %',          f'{pct}%', '6A1B9A', 'F3E5F5'),
    ]
    for i, (label, val, fg, bg) in enumerate(stats, 4):
        # Label cell
        lc            = ws2.cell(row=i, column=2, value=label)
        lc.font       = Font(name='Calibri', size=11, color='495057')
        lc.fill       = PatternFill('solid', fgColor='F8F9FA')
        lc.border     = Border(left=Side(style='thin', color='DEE2E6'),
                               right=Side(style='thin', color='DEE2E6'),
                               top=Side(style='thin',  color='DEE2E6'),
                               bottom=Side(style='thin', color='DEE2E6'))
        lc.alignment  = Alignment(horizontal='left', vertical='center')
        # Value cell
        vc            = ws2.cell(row=i, column=3, value=val)
        vc.font       = Font(name='Calibri', bold=True, size=13, color=fg)
        vc.fill       = PatternFill('solid', fgColor=bg)
        # vc.border     = lc.border
        stat_border = Border(left=Side(style='thin', color='DEE2E6'),
                               right=Side(style='thin', color='DEE2E6'),
                               top=Side(style='thin',  color='DEE2E6'),
                               bottom=Side(style='thin', color='DEE2E6'))
        lc.border = stat_border
        vc.border = stat_border
        vc.alignment  = Alignment(horizontal='center', vertical='center')
        ws2.row_dimensions[i].height = 26

    ws2.column_dimensions['A'].width = 4
    ws2.column_dimensions['B'].width = 28
    ws2.column_dimensions['C'].width = 18

    wb.save(out_path)
    return {'Total reqs': total, 'Matched': matched, 'No match': unmatched, 'Coverage': f'{pct}%'}


# ── Build report ──────────────────────────────────────────────────────────────
print('Building Excel report...')
print('='*65)

STATS = build_report(RESULT_DF, OUTPUT_PATH)

print(f'✓ Report saved → {OUTPUT_PATH}')
print()
for k, v in STATS.items():
    print(f'  {k:<22}: {v}')

Building Excel report...
✓ Report saved → InputFiles\req_test_report.xlsx

  Total reqs            : 83
  Matched               : 29
  No match              : 54
  Coverage              : 34.9%
